For each row arriving from Bronze:

CASE 1: Customer is BRAND NEW (never seen before)
→ INSERT new row
→ is_current = TRUE
→ effective_start_date = modified_date
→ effective_end_date = NULL
→ version_number = 1

CASE 2: Customer EXISTS and location CHANGED (hash different)
→ UPDATE old row: is_current = FALSE, effective_end_date = modified_date
→ INSERT new row: is_current = TRUE, effective_start_date = modified_date
→ version_number = old_version + 1

CASE 3: Customer EXISTS but nothing changed (hash same)
→ DO NOTHING

In [0]:
%python

# ── Cell 2: Imports ───────────────────────────────────────────
%run ../config/pipeline_config

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

print("Libraries loaded ✓")


In [0]:
%python

# ── Cell 3: Read Bronze ───────────────────────────────────────
# STUDY NOTE:
#   We read the full Bronze table — NOT a specific batch path.
#   Bronze table accumulates all batches via APPEND.
#   We filter by modified_date > last Silver watermark
#   to only process rows not yet in Silver.
#
#   Why not read batch path directly?
#   If you read batch_1 directly, Silver never sees batch_2/3.
#   The Bronze TABLE is the single source of truth.

BRONZE_TABLE = "default.bronze_customers"

# Get last Silver processing date
# (separate from Bronze watermark — Silver tracks its own progress)
# FIX: Use modified_date directly instead of effective_start_date timestamp
if spark.catalog.tableExists(SILVER_SCD_TABLE):
    silver_last_processed = spark.table(SILVER_SCD_TABLE) \
        .agg(F.max("modified_date")) \
        .collect()[0][0]
    print(f"Silver last processed: {silver_last_processed}")
    df_bronze = spark.table(BRONZE_TABLE) \
        .filter(F.col("modified_date") > silver_last_processed)
else:
    print("Silver empty — full load from Bronze")
    df_bronze = spark.table(BRONZE_TABLE)

bronze_rows_raw = df_bronze.count()
print(f"Bronze rows (raw): {bronze_rows_raw:,}")

# ★ DEDUPLICATION: Keep only one row per customer (take highest row_id)
# STUDY NOTE:
#   Source data contains duplicate customer records.
#   We must deduplicate before SCD processing to avoid
#   creating multiple version 1 rows for the same customer.
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_unique_id").orderBy(F.desc("row_id"))
df_bronze = df_bronze.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

print(f"Bronze rows (after deduplication): {df_bronze.count():,}")

In [0]:
%python

# ── Cell 4: Add row_hash to Bronze rows ───────────────────────
# STUDY NOTE:
#   row_hash = SHA-256 of the 3 tracked columns joined by '|'
#   Two customers with same city/state/zip → same hash
#   If any of the 3 change → completely different hash
#   This is how we detect changes without comparing each column

df_bronze_hashed = df_bronze.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws("|",
            F.col("customer_zip_code_prefix"),
            F.col("customer_city"),
            F.col("customer_state")
        ), 256
    )
)

print("Row hash added ✓")
print("Sample hashes:")
df_bronze_hashed.select(
    "customer_unique_id", "customer_city",
    "customer_state", "row_hash"
).show(3, truncate=True)

In [0]:
%python

# ── Cell 5: Detect new and changed customers ──────────────────
# STUDY NOTE:
#   We compare incoming Bronze rows to CURRENT Silver rows.
#   "Current" = is_current == TRUE (only one row per customer)
#
#   Three outcomes:
#   left_anti join → customers in Bronze NOT in Silver = NEW
#   inner join + hash diff → customers in both but hash changed = CHANGED
#   inner join + hash same → customers in both, hash same = UNCHANGED (ignore)

silver_exists = spark.catalog.tableExists(SILVER_SCD_TABLE)

if silver_exists:
    df_silver_current = spark.table(SILVER_SCD_TABLE) \
        .filter(F.col("is_current") == True)

    # New customers — never seen before
    df_new_customers = df_bronze_hashed.join(
        df_silver_current.select("customer_unique_id"),
        on="customer_unique_id",
        how="left_anti"
    )

    # Changed customers — exist in Silver but hash is different
    df_changed = df_bronze_hashed.alias("b").join(
        df_silver_current.select("customer_unique_id", "row_hash").alias("s"),
        on="customer_unique_id",
        how="inner"
    ).filter(
        F.col("b.row_hash") != F.col("s.row_hash")
    ).select("b.*")

    df_to_process = df_new_customers.unionByName(df_changed)

else:
    # Initial load — everything is new
    df_to_process = df_bronze_hashed

print(f"Rows to process : {df_to_process.count():,}")

In [0]:
%python

# ── Cell 6: Add version numbers and SCD columns ───────────────
# STUDY NOTE:
#   Before writing to Silver we need to add:
#   customer_key         → surrogate key (unique per row)
#   version_number       → 1 for new, old+1 for changed
#   is_current           → always TRUE for new rows
#   effective_start_date → when this version became active
#   effective_end_date   → NULL (still current)

# Get current max version per customer
if silver_exists:
    df_versions = spark.table(SILVER_SCD_TABLE) \
        .groupBy("customer_unique_id") \
        .agg(F.max("version_number").alias("current_version"))

    df_to_process = df_to_process.join(
        df_versions, on="customer_unique_id", how="left"
    ).withColumn(
        "version_number",
        F.when(F.col("current_version").isNull(), 1)
         .otherwise(F.col("current_version") + 1)
    ).drop("current_version")
else:
    df_to_process = df_to_process.withColumn("version_number", F.lit(1))

# Add all SCD columns
df_new_versions = df_to_process \
    .withColumn("customer_key", F.monotonically_increasing_id()) \
    .withColumn("is_current", F.lit(True)) \
    .withColumn("effective_start_date", F.col("modified_date").cast("timestamp")) \
    .withColumn("effective_end_date", F.lit(None).cast("timestamp")) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

print("SCD columns added ✓")
print(f"New versions ready: {df_new_versions.count():,}")



In [0]:
%python

# ── Cell 7: TWO-STEP SCD Type 2 MERGE ────────────────────────
# STUDY NOTE:
#   SCD Type 2 requires TWO separate operations on Delta:
#
#   STEP 1: EXPIRE old rows
#   For customers who CHANGED — their old Silver row must be
#   marked as expired: is_current=FALSE, end_date=today
#   We match on: customer_unique_id AND is_current=TRUE
#                AND hash is DIFFERENT (changed customers only)
#
#   STEP 2: INSERT new versions
#   All rows in df_new_versions are inserted as new Silver rows.
#   These include both brand new customers AND new versions
#   of changed customers.
#
#   Why two steps and not one MERGE?
#   Delta MERGE can't UPDATE and INSERT the same matched row
#   in one operation. We must expire first, then insert.

if not silver_exists:
    # ── First run: write everything directly ──────────────────
    print("First run — writing full Silver load...")
    df_new_versions.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(SILVER_SCD_TABLE)
    print(f"✓ Silver created: {df_new_versions.count():,} rows")

else:
    # ── Step 1: Expire changed rows ───────────────────────────
    print("Step 1: Expiring changed rows...")

    delta_silver = DeltaTable.forName(spark, SILVER_SCD_TABLE)

    # Only expire rows where customer changed (hash different)
    # Deduplicate to avoid multiple source rows matching the same target
    changed_ids = df_changed.select("customer_unique_id").distinct()

    delta_silver.alias("target").merge(
        changed_ids.alias("source"),
        """
        target.customer_unique_id = source.customer_unique_id
        AND target.is_current = true
        """
    ).whenMatchedUpdate(set={
        "is_current":           F.lit(False),
        "effective_end_date":   F.current_timestamp()
    }).execute()

    print(f"  ✓ Expired rows for {changed_ids.count()} changed customers")

    # ── Step 2: Insert new versions ───────────────────────────
    print("Step 2: Inserting new versions...")

    df_new_versions.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(SILVER_SCD_TABLE)

    print(f"  ✓ Inserted {df_new_versions.count():,} new rows")

In [0]:
%python

# ── Cell 8: Verify SCD Type 2 ─────────────────────────────────
# STUDY NOTE:
#   A healthy SCD Type 2 table should show:
#   - Unchanged customers: 1 row, is_current=TRUE, end_date=NULL
#   - Changed customers:   2+ rows, only 1 with is_current=TRUE
#   - No customer has 2 rows with is_current=TRUE

df_silver = spark.table(SILVER_SCD_TABLE)

print("=" * 55)
print("Silver SCD Type 2 Summary")
print("=" * 55)
print(f"Total rows     : {df_silver.count():,}")
print(f"Current rows   : {df_silver.filter(F.col('is_current')==True).count():,}")
print(f"Expired rows   : {df_silver.filter(F.col('is_current')==False).count():,}")
print("=" * 55)

# Show customers with multiple versions (proof SCD worked)
print("\nCustomers with version history (changed customers):")
display(
    df_silver
    .groupBy("customer_unique_id")
    .agg(F.max("version_number").alias("max_version"))
    .filter(F.col("max_version") > 1)
    .join(df_silver, on="customer_unique_id")
    .select(
        "customer_unique_id", "customer_city",
        "customer_state", "row_hash",
        "version_number", "is_current",
        "effective_start_date", "effective_end_date"
    )
    .orderBy("customer_unique_id", "version_number")
    .limit(20)
)

In [0]:
%python

# Check 1: Does Silver table exist and what's in it?
print(f"Table exists: {spark.catalog.tableExists(SILVER_SCD_TABLE)}")
print(f"Silver path is Delta: {DeltaTable.isDeltaTable(spark, SILVER_PATH)}")
silver_count = spark.table(SILVER_SCD_TABLE).count()
print(f"Silver row count: {silver_count:,}")

# Check 2: What does Bronze table have?
bronze_count = spark.table("default.bronze_customers").count()
print(f"Bronze total rows: {bronze_count:,}")

spark.table("default.bronze_customers") \
    .groupBy("modified_date").count() \
    .orderBy("modified_date").show()

# Check 3: What does watermark say?
spark.table(WATERMARK_TABLE).show(truncate=False)